In [ ]:
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import nltk
from nltk.corpus import wordnet
from nltk.corpus import stopwords
from nltk.corpus.reader import CorpusReader
from nltk.internals import deprecated
from nltk.probability import FreqDist
from nltk.util import binary_search_file as _binary_search_file
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
from textblob import TextBlob

In [ ]:
def get_adjectives(word_list):
    new = []
    for text in word_list:
        new.append(text)
        for syn in wordnet.synsets(text):
            also_sees = syn.also_sees()
            if len(also_sees) > 0:
                for seealso in also_sees:
                    if seealso.pos() in ["a", "s", "r"]:
                        word = seealso.name().split(".")[0]
                        new.append(word)
            similar_tos = syn.similar_tos()
            if len(similar_tos) > 0:
                for similar in similar_tos:
                    if similar.pos() in ["a", "s", "r"]:
                        word = similar.name().split(".")[0]
                        new.append(word)
            attributes = syn.attributes()
            if len(attributes) > 0:
                for attribute in attributes:
                    if attribute.pos() in ["a", "s", "r"]:
                        word = attribute.name().split(".")[0]
    for word in new:
        lemma = lemmatizer.lemmatize(word)
        if lemma not in new:
            new.append(lemma)
    final = []
    for word in new:
        if word not in final:
            final.append(word)
    return final


In [ ]:
df_nyt = pd.read_csv("df_nyt.csv")

f = open("corpus_full.json")
corpus_full = json.load(f)
f.close()

corpus_full_adj = {}

In [ ]:
for key, value in tqdm(corpus_full.items()):
    if key not in corpus_full_adj.keys():
        adjectives = get_adjectives(corpus_full[key]['corpus'])
        corpus_full_adj[key] = adjectives

In [ ]:
# corpus_full_adj

all_adjectives_df = pd.DataFrame.from_dict(corpus_full_adj, orient='index')
all_adjectives_df['adjectives']= all_adjectives_df.values.tolist()

all_adjectives_df = all_adjectives_df[["adjectives"]]
all_adjectives_df = all_adjectives_df.reset_index() 
all_adjectives_df.head()

In [ ]:
all_adjectives_df = all_adjectives_df.explode('adjectives')
all_adjectives_df.head()

In [ ]:
all_adjectives_df.to_csv("adjectives_fullcorpus.csv", index=False)

In [ ]:
f = open("corpus_asian.json")
corpus_asian = json.load(f)
f.close()

corpus_asian_adj = {}

In [ ]:
for key, value in tqdm(corpus_asian.items()):
    if key not in corpus_asian_adj.keys():
        adjectives = get_adjectives(value['corpus'])
        corpus_asian_adj[key] = adjectives

In [ ]:
asian_adjectives_df = pd.DataFrame.from_dict(corpus_asian_adj, orient='index')
asian_adjectives_df['adjectives']= asian_adjectives_df.values.tolist()

asian_adjectives_df = asian_adjectives_df[["adjectives"]]
asian_adjectives_df = asian_adjectives_df.reset_index() 
asian_adjectives_df.head()

In [ ]:
asian_adjectives_df = asian_adjectives_df.explode('adjectives')
asian_adjectives_df.head()

In [ ]:
asian_adjectives_df.to_csv("adjectives_asiancorpus.csv", index=False)

In [ ]:
df_full = pd.read_csv('embeddings_full.csv')
df_full = df_full.sort_values(by="day")